# LoViF 2026 — All-in-One Image Restoration
**Second Challenge on Real-World All-in-One Image Restoration @ ECCV 2026**

Ce notebook exécute le pipeline complet :
1. Installation des dépendances
2. Téléchargement et mise en place des données
3. Exploration et visualisation
4. Entraînement — baseline FoundIR
5. Entraînement — pipeline complet (LoRA + DegradationEncoder + priors physiques)
6. Stage DPO (optimisation par préférences)
7. Évaluation par catégorie (PSNR / SSIM / LPIPS)
8. Inférence finale + soumission (TTA + ensemble)

> **Environnement recommandé** : GPU A100 40 GB (Colab Pro+) ou RTX 3090+.  
> Toutes les cellules sont exécutables de bout en bout.

## 0 — Détection de l'environnement

In [ ]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/LoViF-All-in-One-Image-Restoration'
    !git clone https://github.com/hashirama21/LoViF-All-in-One-Image-Restoration.git {WORK_DIR}
    os.chdir(WORK_DIR)
else:
    WORK_DIR = os.path.dirname(os.path.abspath('lovif2026_pipeline.ipynb'))
    os.chdir(WORK_DIR)

print(f'Working directory: {os.getcwd()}')

## 1 — Installation des dépendances

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
import torchvision

print(f'PyTorch : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 — Téléchargement et mise en place des données

Le dataset LoViF 2026 est distribué via la page officielle du challenge.  
Deux options ci-dessous :
- **Option A** : téléchargement officiel (URL à renseigner quand disponible)
- **Option B** : génération synthétique pour tester le pipeline localement

In [ ]:
# ─── Option A : dataset officiel ────────────────────────────────────────────
# Remplacer LOVIF_URL par le lien de téléchargement officiel quand disponible.
# Format attendu : archive zip contenant train/ validation/ test/

LOVIF_URL = None  # ex: 'https://challenge.example.com/lovif2026_data.zip'

if LOVIF_URL:
    import urllib.request, zipfile
    print('Téléchargement...')
    urllib.request.urlretrieve(LOVIF_URL, 'lovif2026_data.zip')
    print('Extraction...')
    with zipfile.ZipFile('lovif2026_data.zip', 'r') as z:
        z.extractall('data/')
    print('Données extraites dans data/')
else:
    print('LOVIF_URL non défini → Option B (données synthétiques) sera utilisée.')

In [ ]:
# ─── Option B : génération synthétique ──────────────────────────────────────
# Crée un dataset minimal (50 paires par catégorie) pour tester tout le pipeline.

from pathlib import Path
import torch
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np
import sys
sys.path.insert(0, str(Path('.').resolve()))

from src.data.augmentations import (
    MotionBlurDegradation, LowLightDegradation, HazeDegradation,
    RainDegradation, SnowDegradation, GaussianNoiseDegradation
)

CATEGORIES = {
    'blur':      MotionBlurDegradation(),
    'low_light': LowLightDegradation(),
    'haze':      HazeDegradation(),
    'rain':      RainDegradation(),
    'snow':      SnowDegradation(),
}

N_TRAIN = 50   # paires par catégorie (pipeline test)
N_VAL   = 10

def save_tensor(t: torch.Tensor, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    img = Image.fromarray((t.permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8))
    img.save(path)

# Training data
for cat, deg in CATEGORIES.items():
    for i in range(1, N_TRAIN + 1):
        gt = torch.rand(3, 512, 512)                          # image propre synthétique
        lq = deg(gt.clone()).clamp(0, 1)
        name = f'{i:04d}.png'
        save_tensor(gt,  Path(f'data/train/{cat}/gt/{name}'))
        save_tensor(lq,  Path(f'data/train/{cat}/lq/{name}'))
    print(f'  [{cat}] {N_TRAIN} paires générées')

# Validation data (format flat, index-based)
OFFSETS = {'blur': 1, 'low_light': 101, 'haze': 201, 'rain': 301, 'snow': 401}
val_lq  = Path('data/validation/inputs')
val_gt  = Path('data/validation/gt')
for cat, deg in CATEGORIES.items():
    offset = OFFSETS[cat]
    for i in range(N_VAL):
        gt = torch.rand(3, 512, 512)
        lq = deg(gt.clone()).clamp(0, 1)
        name = f'{offset + i:04d}.png'
        save_tensor(gt, val_gt  / name)
        save_tensor(lq, val_lq  / name)

print(f'\nDataset synthétique créé : {N_TRAIN} paires/catégorie (train), {N_VAL}/catégorie (val)')

## 3 — Exploration et visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
import numpy as np

CATEGORIES = ['blur', 'low_light', 'haze', 'rain', 'snow']

fig = plt.figure(figsize=(14, 3.2 * len(CATEGORIES)))
gs  = gridspec.GridSpec(len(CATEGORIES), 3, hspace=0.35, wspace=0.05)

for row, cat in enumerate(CATEGORIES):
    lq_files = sorted(Path(f'data/train/{cat}/lq').glob('*.png'))
    gt_files = sorted(Path(f'data/train/{cat}/gt').glob('*.png'))
    if not lq_files:
        continue
    lq_img = np.array(Image.open(lq_files[0]).resize((256,256)))
    gt_img = np.array(Image.open(gt_files[0]).resize((256,256)))

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.imshow(gt_img); ax0.set_title(f'{cat.upper()} — GT', fontsize=9); ax0.axis('off')

    ax1 = fig.add_subplot(gs[row, 1])
    ax1.imshow(lq_img); ax1.set_title(f'{cat.upper()} — LQ (dégradé)', fontsize=9); ax1.axis('off')

    diff = np.abs(gt_img.astype(int) - lq_img.astype(int)).astype(np.uint8)
    ax2 = fig.add_subplot(gs[row, 2])
    ax2.imshow(diff * 3); ax2.set_title('|GT - LQ| ×3', fontsize=9); ax2.axis('off')

plt.suptitle('LoViF 2026 — aperçu par catégorie', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Aperçu sauvegardé : data_overview.png')

In [ ]:
# Distribution du dataset
from collections import Counter
import matplotlib.pyplot as plt

counts = {}
for cat in CATEGORIES:
    n = len(list(Path(f'data/train/{cat}/lq').glob('*.png')))
    counts[cat] = n

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(counts.keys(), counts.values(), color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'])
ax.bar_label(bars)
ax.set_ylabel('Nombre de paires LQ/GT')
ax.set_title('Distribution du dataset d\'entraînement')
plt.tight_layout()
plt.show()
print(f"Total paires : {sum(counts.values())}")

## 4 — Preprocessing : vérification du pipeline de données

In [ ]:
from src.data.dataset import LoViFDataset, LoViFValDataset
from torch.utils.data import DataLoader

train_ds = LoViFDataset('data/train', composite_prob=0.35, augment=True)
val_ds   = LoViFValDataset('data/validation/inputs', has_gt=True)

print(f'Taille entraînement : {len(train_ds)} paires')
print(f'Taille validation   : {len(val_ds)} images')

# Vérification d'un batch
loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
batch  = next(iter(loader))

print(f'\nBatch shapes :')
print(f'  lq : {batch["lq"].shape}  [{batch["lq"].min():.3f}, {batch["lq"].max():.3f}]')
print(f'  gt : {batch["gt"].shape}  [{batch["gt"].min():.3f}, {batch["gt"].max():.3f}]')
print(f'  catégories : {batch["category"]}')

In [ ]:
# Visualiser l'effet de l'augmentation composite
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from src.data.augmentations import CompositeDegradationPipeline
import torch

sample = train_ds[0]
gt_t   = sample['gt']
pipeline = CompositeDegradationPipeline(composite_prob=1.0)  # toujours composite

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(gt_t.permute(1,2,0).numpy()); axes[0].set_title('GT original'); axes[0].axis('off')
for i in range(1, 4):
    aug = pipeline(gt_t.clone())
    axes[i].imshow(aug.permute(1,2,0).numpy())
    axes[i].set_title(f'Composite aug #{i}')
    axes[i].axis('off')
plt.suptitle('Pipeline de dégradation composite')
plt.tight_layout()
plt.show()

In [ ]:
# Vérification des priors physiques
import torch
from src.models.physical_priors import RetinexPrior, DarkChannelPrior

retinex   = RetinexPrior(sigma=31)
dark_ch   = DarkChannelPrior()

# Cas test : image simulant le brouillard (faible contraste)
hazy = train_ds[next(i for i, (lp,gp,c) in enumerate(train_ds.pairs) if c=='haze')]['lq'].unsqueeze(0)

illum, reflect   = retinex(hazy)
transmission     = dark_ch(hazy)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, t, title in zip(axes,
    [hazy.squeeze(0), illum.squeeze(), reflect.squeeze(0), transmission.squeeze()],
    ['LQ Haze', 'Illumination (Retinex)', 'Reflectance', 'Transmission (DCP)']):
    img = t.permute(1,2,0).numpy() if t.ndim == 3 else t.numpy()
    ax.imshow(img.clip(0,1), cmap='gray' if img.ndim==2 else None)
    ax.set_title(title, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 5 — Entraînement : Baseline FoundIR

In [ ]:
# Générer une config baseline minimale (override pour le notebook)
from omegaconf import OmegaConf

baseline_cfg = OmegaConf.create({
    'project': {
        'name': 'lovif2026',
        'run_name': 'baseline_notebook',
        'seed': 42,
        'output_dir': './outputs/baseline_notebook',
    },
    'data': {
        'train_dir': './data/train',
        'val_dir': './data/validation/inputs',
        'batch_size': 2,
        'num_workers': 0,
        'composite_prob': 0.0,
    },
    'model': {
        'lora_rank': 0,
        'use_degradation_encoder': False,
        'use_physical_priors': False,
        'encoder_dim': 512,
    },
    'training': {
        'max_steps': 100,           # court pour le notebook — augmenter pour la compét.
        'lr': 1e-4,
        'warmup_steps': 10,
        'gradient_accumulation_steps': 2,
        'mixed_precision': 'bf16',
        'gradient_checkpointing': True,
        'weight_decay': 0.01,
        'save_every': 50,
        'eval_every': 50,
        'early_stopping_patience': 10,
    },
    'loss': {
        'l1_weight': 1.0,
        'lpips_weight': 0.0,
        'adversarial_weight': 0.0,
        'pixel_loss_weight': 0.0,
    },
    'logging': {'use_wandb': False},
})

print(OmegaConf.to_yaml(baseline_cfg))

In [ ]:
import random, numpy as np, torch

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# ⚠️  Cette cellule télécharge FoundIR (~5 GB) à la première exécution.
# Sur Colab, utilisez un cache Drive pour éviter le re-téléchargement.

from src.training import Trainer

trainer_baseline = Trainer(baseline_cfg)
print(f'Modèle instancié sur {trainer_baseline.device}')

n_trainable = sum(p.numel() for p in trainer_baseline.model.trainable_parameters())
n_total     = sum(p.numel() for p in trainer_baseline.model.parameters())
print(f'Paramètres entraînables : {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)')

In [ ]:
trainer_baseline.fit()
print('\nEntraînement baseline terminé.')

## 6 — Entraînement : Pipeline complet (LoRA + DegradationEncoder + Priors physiques)

In [ ]:
from omegaconf import OmegaConf

full_cfg = OmegaConf.create({
    'project': {
        'name': 'lovif2026',
        'run_name': 'full_pipeline_notebook',
        'seed': 42,
        'output_dir': './outputs/full_pipeline_notebook',
    },
    'data': {
        'train_dir': './data/train',
        'val_dir': './data/validation/inputs',
        'batch_size': 1,
        'num_workers': 0,
        'composite_prob': 0.35,
    },
    'model': {
        'lora_rank': 32,
        'lora_alpha': 64,
        'lora_dropout': 0.1,
        'use_degradation_encoder': True,
        'encoder_dim': 512,
        'use_physical_priors': True,
    },
    'training': {
        'max_steps': 200,           # augmenter à 80000 pour la compétition
        'lr': 5e-5,
        'warmup_steps': 20,
        'gradient_accumulation_steps': 4,
        'mixed_precision': 'bf16',
        'gradient_checkpointing': True,
        'weight_decay': 0.01,
        'save_every': 100,
        'eval_every': 100,
        'early_stopping_patience': 10,
    },
    'loss': {
        'l1_weight': 1.0,
        'lpips_weight': 0.15,
        'adversarial_weight': 0.0,  # désactivé pour la rapidité du test
        'pixel_loss_weight': 0.1,
    },
    'logging': {'use_wandb': False},
})

set_seed(42)
trainer_full = Trainer(full_cfg)

n_trainable = sum(p.numel() for p in trainer_full.model.trainable_parameters())
n_total     = sum(p.numel() for p in trainer_full.model.parameters())
print(f'LoRA + encoder — paramètres entraînables : {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)')

In [ ]:
trainer_full.fit()
print('\nEntraînement pipeline complet terminé.')

## 7 — Stage DPO : Optimisation par préférences

In [ ]:
# Générer des paires DPO synthétiques à partir du checkpoint full
from pathlib import Path
import torch
from src.models import RestorationPipeline, PipelineConfig
from src.losses.dpo import RewardModel

CKPT_PATH = './outputs/full_pipeline_notebook/best.ckpt'

if not Path(CKPT_PATH).exists():
    # Fallback : utiliser le checkpoint périodique
    ckpts = sorted(Path('./outputs/full_pipeline_notebook').glob('*.pt'))
    if ckpts:
        CKPT_PATH = str(ckpts[-1])
        print(f'best.ckpt absent — utilisation de {CKPT_PATH}')
    else:
        CKPT_PATH = None
        print('Aucun checkpoint trouvé. Exécuter les cellules d\'entraînement d\'abord.')

if CKPT_PATH:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    state  = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    cfg_m  = state.get('model_config', {})
    model  = RestorationPipeline(PipelineConfig(**cfg_m) if cfg_m else PipelineConfig()).to(device)
    model.load_state_dict(state['model_state'], strict=False)
    model.eval()
    print(f'Modèle chargé depuis {CKPT_PATH}')

In [ ]:
# Générer N restorations pour chaque image, sélectionner best/worst
from src.losses.dpo import RewardModel
from src.data.dataset import LoViFValDataset, _resize_512
import numpy as np
from PIL import Image

if CKPT_PATH:
    reward_model  = RewardModel(lpips_weight=1.0, musiq_weight=0.0, clip_iqa_weight=0.0)
    N_GEN         = 3   # générations par image (augmenter pour prod)
    N_DPO_IMAGES  = 5   # images de validation à utiliser

    dpo_lq_dir  = Path('data/dpo_pairs/lq')
    dpo_cho_dir = Path('data/dpo_pairs/chosen')
    dpo_rej_dir = Path('data/dpo_pairs/rejected')
    for d in [dpo_lq_dir, dpo_cho_dir, dpo_rej_dir]:
        d.mkdir(parents=True, exist_ok=True)

    val_files = sorted(Path('data/validation/inputs').glob('*.png'))[:N_DPO_IMAGES]

    def tensor_to_pil(t):
        return Image.fromarray((t.squeeze(0).permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8))

    for lq_path in val_files:
        from torchvision import transforms
        lq_t = _resize_512(transforms.ToTensor()(Image.open(lq_path).convert('RGB'))).unsqueeze(0).to(device)
        gt_t = _resize_512(transforms.ToTensor()(Image.open(
            Path('data/validation/gt') / lq_path.name).convert('RGB'))).unsqueeze(0).to(device)

        with torch.inference_mode():
            gens = [model.restore(lq_t) for _ in range(N_GEN)]

        chosen, rejected = reward_model.select_pairs(gens, gt_t)

        tensor_to_pil(lq_t).save(dpo_lq_dir  / lq_path.name)
        tensor_to_pil(chosen).save(dpo_cho_dir / lq_path.name)
        tensor_to_pil(rejected).save(dpo_rej_dir / lq_path.name)

    print(f'{len(val_files)} triplets DPO générés dans data/dpo_pairs/')

In [ ]:
from omegaconf import OmegaConf
from src.training import DPOTrainer

if CKPT_PATH:
    dpo_cfg = OmegaConf.create({
        'project': {
            'name': 'lovif2026',
            'run_name': 'dpo_notebook',
            'seed': 42,
            'output_dir': './outputs/dpo_notebook',
        },
        'model': {
            'lora_rank': 32,
            'use_degradation_encoder': True,
            'encoder_dim': 512,
            'use_physical_priors': True,
        },
        'dpo': {
            'base_checkpoint': CKPT_PATH,
            'preferences_dir': './data/dpo_pairs',
            'beta': 0.1,
            'lr': 2e-6,
            'warmup_steps': 5,
            'max_steps': 20,             # augmenter à 3000 pour la compétition
            'batch_size': 1,
            'gradient_accumulation_steps': 2,
        },
        'data': {'val_dir': './data/validation/inputs'},
        'logging': {'use_wandb': False},
    })

    set_seed(42)
    dpo_trainer = DPOTrainer(dpo_cfg)
    dpo_trainer.fit()
    print('Stage DPO terminé.')

## 8 — Évaluation par catégorie (PSNR / SSIM / LPIPS)

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from src.data.dataset import LoViFValDataset
from src.models import RestorationPipeline, PipelineConfig
from src.utils import MetricBag

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Chercher le meilleur checkpoint disponible
def find_best_ckpt(run_dir):
    best = Path(run_dir) / 'best.ckpt'
    if best.exists():
        return str(best)
    ckpts = sorted(Path(run_dir).glob('*.pt'))
    return str(ckpts[-1]) if ckpts else None

def evaluate(ckpt_path, val_dir, device):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg   = state.get('model_config', {})
    model = RestorationPipeline(PipelineConfig(**cfg) if cfg else PipelineConfig()).to(device)
    model.load_state_dict(state['model_state'], strict=False)
    model.eval()

    ds     = LoViFValDataset(val_dir, has_gt=True)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    bag    = MetricBag()

    with torch.inference_mode():
        for batch in tqdm(loader, desc='Évaluation', leave=False):
            lq  = batch['lq'].to(device)
            gt  = batch['gt'].to(device)
            cat = batch['category'][0]
            with torch.autocast(device.type, dtype=torch.bfloat16):
                pred = model.restore(lq)
            bag.update(pred.float(), gt.float(), category=cat)

    return bag.summary()

RUNS = {
    'Baseline':       find_best_ckpt('./outputs/baseline_notebook'),
    'Full pipeline':  find_best_ckpt('./outputs/full_pipeline_notebook'),
    'DPO':            find_best_ckpt('./outputs/dpo_notebook'),
}
VAL_DIR = './data/validation/inputs'

results = {}
for name, ckpt in RUNS.items():
    if ckpt is None:
        print(f'[{name}] aucun checkpoint — ignoré')
        continue
    print(f'\nÉvaluation : {name}  ({ckpt})')
    results[name] = evaluate(ckpt, VAL_DIR, device)

In [ ]:
# Tableau de résultats
import pandas as pd

CATS = ['blur', 'low_light', 'haze', 'rain', 'snow', 'all']
rows = []
for run_name, summary in results.items():
    for cat in CATS:
        if cat not in summary:
            continue
        v = summary[cat]
        rows.append({
            'Run': run_name, 'Catégorie': cat,
            'PSNR ↑': round(v['psnr'],  2),
            'SSIM ↑': round(v['ssim'],  4),
            'LPIPS ↓': round(v['lpips'], 4),
            'N': int(v['n']),
        })

df = pd.DataFrame(rows).set_index(['Run', 'Catégorie'])
print(df.to_string())

In [ ]:
# Comparaison visuelle : LQ → Baseline → Full → DPO → GT
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms
from src.data.dataset import _resize_512

val_files = sorted(Path(VAL_DIR).glob('*.png'))[:5]
col_labels = ['LQ (entrée)'] + list(results.keys()) + ['GT (référence)']

fig, axes = plt.subplots(len(val_files), len(col_labels),
                          figsize=(3.5 * len(col_labels), 3.5 * len(val_files)))

# Charger les modèles une seule fois
loaded_models = {}
for run_name, ckpt in RUNS.items():
    if ckpt is None: continue
    state = torch.load(ckpt, map_location=device, weights_only=False)
    cfg   = state.get('model_config', {})
    m = RestorationPipeline(PipelineConfig(**cfg) if cfg else PipelineConfig()).to(device)
    m.load_state_dict(state['model_state'], strict=False)
    m.eval()
    loaded_models[run_name] = m

to_tensor = transforms.ToTensor()

for row, lq_path in enumerate(val_files):
    lq_t = _resize_512(to_tensor(Image.open(lq_path).convert('RGB'))).unsqueeze(0).to(device)
    gt_path = Path('data/validation/gt') / lq_path.name
    gt_t = _resize_512(to_tensor(Image.open(gt_path).convert('RGB')))

    col = 0
    axes[row][col].imshow(lq_t.squeeze(0).permute(1,2,0).cpu().numpy().clip(0,1))
    axes[row][col].set_title(col_labels[col] if row == 0 else '', fontsize=8)
    axes[row][col].axis('off')
    col += 1

    for run_name, m in loaded_models.items():
        with torch.inference_mode():
            with torch.autocast(device.type, dtype=torch.bfloat16):
                pred = m.restore(lq_t).squeeze(0).permute(1,2,0).cpu().numpy().clip(0,1)
        axes[row][col].imshow(pred)
        axes[row][col].set_title(col_labels[col] if row == 0 else '', fontsize=8)
        axes[row][col].axis('off')
        col += 1

    axes[row][col].imshow(gt_t.permute(1,2,0).numpy().clip(0,1))
    axes[row][col].set_title(col_labels[col] if row == 0 else '', fontsize=8)
    axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('comparison_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Comparaison sauvegardée : comparison_results.png')

## 9 — Inférence finale : TTA + Ensemble → soumission challenge

In [ ]:
from omegaconf import OmegaConf
from pathlib import Path

# Construire la liste des checkpoints disponibles
available_ckpts = []
available_weights = []

for run_dir, w in [('./outputs/full_pipeline_notebook', 0.45),
                   ('./outputs/dpo_notebook', 0.55)]:
    ckpt = find_best_ckpt(run_dir)
    if ckpt:
        available_ckpts.append(ckpt)
        available_weights.append(w)

# Normaliser les poids
w_sum = sum(available_weights)
available_weights = [w / w_sum for w in available_weights]

if not available_ckpts:
    print('Aucun checkpoint disponible — exécuter les étapes d\'entraînement.')
else:
    print(f'Checkpoints pour l\'ensemble :')
    for ck, wt in zip(available_ckpts, available_weights):
        print(f'  {ck}  (poids {wt:.2f})')

    infer_cfg = OmegaConf.create({
        'inference': {
            'checkpoints':       available_ckpts,
            'ensemble_weights':  available_weights,
            'num_inference_steps': 25,
            'guidance_scale':    7.5,
            'device':            'cuda' if torch.cuda.is_available() else 'cpu',
            'tta': {'enabled': True},
            'input_dir':  './data/validation/inputs',
            'output_dir': './validation_outputs',
        }
    })

In [ ]:
from src.inference import InferenceEngine

if available_ckpts:
    engine = InferenceEngine(infer_cfg)
    engine.run(
        input_dir  = infer_cfg.inference.input_dir,
        output_dir = infer_cfg.inference.output_dir,
    )
    out_files = sorted(Path(infer_cfg.inference.output_dir).glob('*.png'))
    print(f'\n{len(out_files)} images générées dans {infer_cfg.inference.output_dir}')

In [ ]:
# Aperçu des sorties finales
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

out_dir = Path('./validation_outputs')
in_dir  = Path('./data/validation/inputs')

out_files = sorted(out_dir.glob('*.png'))[:5]
if out_files:
    fig, axes = plt.subplots(len(out_files), 2, figsize=(8, 3 * len(out_files)))
    if len(out_files) == 1:
        axes = [axes]

    for row, out_path in enumerate(out_files):
        lq  = np.array(Image.open(in_dir / out_path.name).resize((256,256)))
        res = np.array(Image.open(out_path).resize((256,256)))
        axes[row][0].imshow(lq);  axes[row][0].set_title('LQ', fontsize=9);  axes[row][0].axis('off')
        axes[row][1].imshow(res); axes[row][1].set_title(f'Restauré — {out_path.name}', fontsize=9); axes[row][1].axis('off')

    plt.suptitle('Résultats finaux (TTA + Ensemble)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('final_outputs.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Aperçu final sauvegardé : final_outputs.png')

## 10 — Récapitulatif des résultats & recommandations compétition

In [ ]:
print('=' * 60)
print('RÉCAPITULATIF PIPELINE LoViF 2026')
print('=' * 60)

if results:
    print(df[df.index.get_level_values('Catégorie') == 'all'].to_string())

print()
print('RECOMMANDATIONS POUR LA COMPÉTITION :')
recs = [
    ('max_steps (full)',    '80 000 steps    (configs/train_full.yaml)'),
    ('max_steps (DPO)',     '3 000 steps     (configs/dpo.yaml)'),
    ('batch_size',         '4 × grad_accum=4 → batch effectif 16'),
    ('lora_rank',          '32 (ou 64 si VRAM suffisante)'),
    ('num_inference_steps','25 (inférence) / 4 (training proxy)'),
    ('TTA',                '4-way (hflip + vflip + rot90)'),
    ('Ensemble',           '2 checkpoints : full + DPO, poids 0.45/0.55'),
]
for param, val in recs:
    print(f'  {param:<26} {val}')

print()
print('Pour lancer l\'entraînement complet :')
print('  python scripts/train.py                          # full pipeline')
print('  python scripts/train.py --config-name=train_baseline')
print('  python scripts/dpo_stage.py')
print('  python scripts/infer.py')
print('=' * 60)